# 02 — First baseline and submission

最小のend-to-endパイプラインを作ります。固定したStratifiedKFoldでOOFを作り、
同じfoldを後続モデルでも共有します。前処理はPipeline内にあるためvalidation情報はfitに入りません。

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from config import Baseline
from features import get_base_features
from train import make_model, run_cv
from validation import add_stratified_folds

TARGET = Baseline.TARGET
ID_COLUMN = Baseline.ID_COLUMN
FOLD_COLUMN = Baseline.FOLD_COLUMN

train = pd.read_csv(ROOT / "input" / "train.csv")
test = pd.read_csv(ROOT / "input" / "test.csv")
train[TARGET] = train[TARGET].map({"Yes": 1, "No": 0})
assert train[TARGET].notna().all()
print("train:", train.shape, "test:", test.shape)

## 特徴量とモデル

比較可能性を優先し、このXGBoost定義をNotebook 04でもそのまま使います。

In [ ]:
BASE_FEATURES, BASE_NUM, BASE_CAT = get_base_features(
    train, TARGET, ID_COLUMN, min_nunique=Baseline.MIN_NUMERIC_UNIQUE
)
model = make_model(
    "XGBoost", BASE_NUM, BASE_CAT, seed=Baseline.SEED, use_gpu=False
)
print("features:", len(BASE_FEATURES), "numeric:", len(BASE_NUM), "categorical:", len(BASE_CAT))
model

## foldを一度だけ作って保存

全モデルで同じvalidation行を使うため、IDとfoldの対応をCSVとして固定します。

In [ ]:
train = add_stratified_folds(
    train,
    TARGET,
    n_splits=Baseline.N_SPLITS,
    random_state=Baseline.SEED,
    fold_column=FOLD_COLUMN,
)
fold_path = ROOT / "output" / "stkfolds.csv"
fold_path.parent.mkdir(parents=True, exist_ok=True)
train[[ID_COLUMN, FOLD_COLUMN]].to_csv(fold_path, index=False)
display(train.groupby(FOLD_COLUMN)[TARGET].agg(["size", "mean"]))

## OOF評価とtest予測

採用指標はROC AUCです。各foldのモデルでtestを予測し、その平均を提出値にします。

In [ ]:
result = run_cv(
    model=model,
    train=train,
    test=test,
    features=BASE_FEATURES,
    target=TARGET,
    id_column=ID_COLUMN,
    fold_column=FOLD_COLUMN,
    label="first_xgboost",
    save_prefix="first_xgboost",
    output_dir=ROOT / "artifacts",
)

## submission

予測確率の範囲・行数・ID順を検証してから保存します。

In [ ]:
submission = pd.DataFrame(
    {ID_COLUMN: test[ID_COLUMN].to_numpy(), TARGET: result["test_pred"]}
)
assert len(submission) == len(test)
assert submission[ID_COLUMN].equals(test[ID_COLUMN])
assert submission[TARGET].between(0, 1).all()

submission_path = ROOT / "submission.csv"
submission.to_csv(submission_path, index=False)
print("saved:", submission_path)
display(submission.head())